# MATLAB–Python operational parity
Change only `CASE_NAME`, or populate `CASE_NAMES` to run several fixtures.


In [ ]:
CASE_NAME = "_simp"
CASE_NAMES = []  # Example: ["_simp", "two_zone_interzone"]


In [ ]:
from pathlib import Path
import json, shutil, subprocess, sys
import pandas as pd

REPO = Path.cwd().resolve()
while not (REPO / "src/brcm").is_dir():
    REPO = REPO.parent
MATLAB = shutil.which("matlab")
CASES = CASE_NAMES or [CASE_NAME]

def run(command):
    result = subprocess.run(command, cwd=REPO, text=True, capture_output=True)
    print(result.stdout, result.stderr)
    return result

for case_name in CASES:
    if MATLAB:
        matlab_case = case_name.replace("'", "''")
        run([MATLAB, "-batch", f"addpath(fullfile(pwd,'origin_matlab','parity')); export_case_reference('{matlab_case}')"])
    else:
        print(f"MATLAB unavailable; retaining any existing reference for {case_name}")
    run([sys.executable, "pre_test/tools/parity/run_python_reference.py", "--case", case_name])
    run([sys.executable, "pre_test/tools/parity/compare_parity.py", "--case", case_name])


In [ ]:
detail_rows, summary_rows = [], []
for case_name in CASES:
    report_path = REPO / "pre_test/outputs/parity" / case_name / "parity_report.json"
    report = json.loads(report_path.read_text())
    checks = {
        "Same IDF": report["same_input_file"],
        "Tables": all(item["pass"] for item in report["tables"].values()),
        "IDs": report["identifiers"]["pass"],
        "Boundaries": report["boundaries"]["pass"],
        "A": report["matrices"].get("A", {}).get("pass", False),
        "Bq": report["matrices"].get("Bq", {}).get("pass", False),
        "Xcap": report["matrices"].get("Xcap", {}).get("pass", False),
        "Simulation": report["simulation"]["pass"],
    }
    for check, passed in checks.items():
        metric = report["matrices"].get(check, {})
        detail_rows.append({"Case": case_name, "Check": check, "Result": "PASS" if passed else "FAIL",
                            "MATLAB": metric.get("matlab_shape", report["matlab_sha256"] if check == "Same IDF" else "—"),
                            "Python": metric.get("python_shape", report["python_sha256"] if check == "Same IDF" else "—"),
                            "Difference": metric.get("max_absolute_error", report.get("first_mismatch") if not passed else 0)})
    summary_rows.append({"Case": case_name, **{name: "PASS" if value else "FAIL" for name, value in checks.items()},
                         "Overall": "PASS" if report["overall_pass"] else "FAIL"})

display(pd.DataFrame(detail_rows, columns=["Case", "Check", "Result", "MATLAB", "Python", "Difference"]))
display(pd.DataFrame(summary_rows))
